In [12]:
!pip install torch torchvision opencv-python scikit-image pandas matplotlib


In [13]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from skimage.exposure import match_histograms


In [14]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 128
NORMAL_CLASS = 0   # airplane = normal


In [15]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


In [16]:
ref_tensor, _ = dataset[0]
reference_img = (ref_tensor.permute(1,2,0).numpy() * 255).astype(np.uint8)


In [17]:
def preprocess_image(img, ref_img):
    img = cv2.resize(img, (ref_img.shape[1], ref_img.shape[0]))

    # Lighting normalization
    img = match_histograms(img, ref_img, channel_axis=-1)

    # Feature enhancement
    gray = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(3.0, (8,8))
    enhanced = clahe.apply(gray)

    return enhanced


In [18]:
class FaultClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [19]:
class FaultLocalizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 1, 2, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


In [20]:
class InspectionController:
    def __init__(self, model_a, model_b):
        self.model_a = model_a.to(DEVICE).eval()
        self.model_b = model_b.to(DEVICE).eval()
        self.logs = []

    def inspect(self, idx, img, actual_label):
        entry = {"Index": idx}

        # Stage 1 — classification
        t0 = time.perf_counter()
        with torch.no_grad():
            fault_prob = self.model_a(img).item()
        entry["Classifier_ms"] = round((time.perf_counter()-t0)*1000,2)

        if fault_prob > 0.5:
            # Stage 2 — localization
            t1 = time.perf_counter()
            with torch.no_grad():
                _ = self.model_b(img)
            entry["Localizer_ms"] = round((time.perf_counter()-t1)*1000,2)
            entry["Decision"] = "FAULT"
        else:
            entry["Localizer_ms"] = 0.0
            entry["Decision"] = "PASS"

        entry["Actual"] = "FAULT" if actual_label else "NORMAL"
        entry["Total_ms"] = entry["Classifier_ms"] + entry["Localizer_ms"]
        self.logs.append(entry)


In [21]:
model_a = FaultClassifier()
model_b = FaultLocalizer()

controller = InspectionController(model_a, model_b)

for i in range(20):  # simulate batch inspection
    img_tensor, label = dataset[i]

    img_np = (img_tensor.permute(1,2,0).numpy() * 255).astype(np.uint8)
    processed = preprocess_image(img_np, reference_img)

    img_t = torch.tensor(processed).unsqueeze(0).unsqueeze(0).float().to(DEVICE) / 255.0

    actual_fault = 0 if label == NORMAL_CLASS else 1
    controller.inspect(i, img_t, actual_fault)


In [23]:
report = pd.DataFrame(controller.logs)

print("="*80)
print("PRODUCTION VISUAL INSPECTION REPORT")
print("="*80)
print(report)
print("="*80)
print("Average Latency (ms):", report["Total_ms"].mean())


PRODUCTION VISUAL INSPECTION REPORT
    Index  Classifier_ms  Localizer_ms Decision  Actual  Total_ms
0       0           3.48           0.0     PASS   FAULT      3.48
1       1           3.27           0.0     PASS   FAULT      3.27
2       2           3.34           0.0     PASS   FAULT      3.34
3       3           3.48           0.0     PASS  NORMAL      3.48
4       4          11.72           0.0     PASS   FAULT     11.72
5       5           9.72           0.0     PASS   FAULT      9.72
6       6           5.53           0.0     PASS   FAULT      5.53
7       7           9.76           0.0     PASS   FAULT      9.76
8       8          15.79           0.0     PASS   FAULT     15.79
9       9          15.73           0.0     PASS   FAULT     15.73
10     10           8.48           0.0     PASS  NORMAL      8.48
11     11          10.72           0.0     PASS   FAULT     10.72
12     12           6.76           0.0     PASS   FAULT      6.76
13     13           3.46           0.0  